In [1]:
import numpy as np
import time
import random
import os

def readVectorsSeq(filename):
    """
    Reads dataset file and returns list of numpy arrays (vectors)
    """
    P = []
    
    with open(filename, 'r') as f:
        for line in f:
            values = list(map(float, line.strip().split(',')))
            P.append(np.array(values))
    
    print(f"Loaded {len(P)} points with dimension {len(P[0])}")
    return P

def sqdist(a, b):
    return np.sum((a - b) ** 2)

def kcenter(P, k):
    """
    Farthest First Traversal Algorithm
    Time Complexity: O(n * k)
    """
    n = len(P)
    
    # Step 1: pick random first center
    centers = [P[random.randint(0, n - 1)]]
    
    # Step 2: initialize distances
    dist = np.array([sqdist(p, centers[0]) for p in P])
    
    for _ in range(1, k):
        # pick farthest point
        idx = np.argmax(dist)
        new_center = P[idx]
        centers.append(new_center)
        
        # update distances
        for i in range(n):
            dist[i] = min(dist[i], sqdist(P[i], new_center))
    
    return centers

def kmeansPP(P, k):
    """
    K-Means++ Initialization
    Time Complexity: O(n * k)
    """
    n = len(P)
    
    # Step 1: random first center
    centers = [P[random.randint(0, n - 1)]]
    
    # Step 2: initialize distances
    dist = np.array([sqdist(p, centers[0]) for p in P])
    
    for _ in range(1, k):
        # probability distribution proportional to distance squared
        prob = dist / dist.sum()
        
        # choose new center
        idx = np.random.choice(n, p=prob)
        new_center = P[idx]
        centers.append(new_center)
        
        # update distances
        for i in range(n):
            dist[i] = min(dist[i], sqdist(P[i], new_center))
    
    return centers

def kmeansObj(P, C):
    """
    Computes average squared distance to closest center
    """
    total = 0.0
    
    for p in P:
        min_dist = float('inf')
        for c in C:
            d = sqdist(p, c)
            if d < min_dist:
                min_dist = d
        total += min_dist
    
    return total / len(P)

In [4]:
# -------- INPUT PARAMETERS --------
file_path = "../datasets/Q1- UCI Spam clustering/spambase.data"

k = 10       # smaller number of clusters
k1 = 50      # larger coreset size (must be > k)

# -------- LOAD DATA --------
P = readVectorsSeq(file_path)

start = time.time()
C_kcenter = kcenter(P, k)
end = time.time()

start = time.time()
C_kmeanspp = kmeansPP(P, k)
obj_val = kmeansObj(P, C_kmeanspp)
end = time.time()



# Step 1: get k1 centers
start = time.time()
X = kcenter(P, k1)
mid = time.time()

# Step 2: reduce to k centers
C_final = kmeansPP(X, k)

# Step 3: compute objective on full dataset
obj_val2 = kmeansObj(P, C_final)
end = time.time()


print("\n================================================")

print("\n[Task 1] Farthest-First Traversal")
print(f"Number of centers (k): {k}")
print(f"Runtime: {end - start:.4f} seconds")

print("\n[Task 2] k-means++ on full dataset")
print(f"Objective value: {obj_val:.4f}")

print("\n[Task 3] Coreset experiment")
print(f"k1 (coreset size): {k1}")
print(f"Final objective value: {obj_val2:.4f}")

print("\n================================================")

Loaded 4601 points with dimension 58


[Task 1] Farthest-First Traversal
Number of centers (k): 10
Runtime: 4.2223 seconds

[Task 2] k-means++ on full dataset
Objective value: 27713.3253

[Task 3] Coreset experiment
k1 (coreset size): 50
Final objective value: 55498.3264

